In [8]:
# Step 1 - Initialize the Spark session for the Silver transformation layer
# This notebook reads the Bronze Iceberg table, standardizes the dataset,
# applies data quality rules, and writes the result into the Silver layer.

import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark

spark = get_spark("silver-transformations")

In [9]:
# Step 2 - Read the Bronze sales table from the Nessie catalog
# The Silver layer is built from the Bronze layer, not directly from the raw CSV.

sales_bronze = spark.table("nessie.bronze.sales")
sales_bronze.show(5, truncate=False)
sales_bronze.printSchema()

+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name|Segment    |Country      |City        |State       |Postal Code|Region |Product ID     |Category       |Sub-Category|Product Name                                                            |Sales  |Quantity|Discount|Profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+-------------+-----------+-------------+------------+------------+-----------+-------+---------------+---------------+------------+------------------

In [10]:
# Step 3 - Standardize the Bronze dataset into the Silver layer
# This step renames the source columns, preserves ingestion metadata,
# and applies light normalization and data quality filters.

from pyspark.sql.functions import col, trim

sales_silver = (
    sales_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("source_file"),
        col("source_system"),
        col("batch_id")
    )
    .withColumn("ship_mode", trim(col("ship_mode")))
    .withColumn("segment", trim(col("segment")))
    .withColumn("country", trim(col("country")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("region", trim(col("region")))
    .withColumn("category", trim(col("category")))
    .withColumn("sub_category", trim(col("sub_category")))
    .withColumn("product_name", trim(col("product_name")))
    .dropna(subset=["order_id", "product_id", "sales"])
    .dropDuplicates(["order_id", "product_id"])
)

sales_silver.show(5, truncate=False)
sales_silver.printSchema()

+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name   |segment    |country      |city         |state     |postal_code|region|product_id     |category       |sub_category|product_name                                                  |sales  |quantity|discount|profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------

In [12]:
# Step 4 - Run data quality checks on the Silver dataset
# These checks validate the row counts, null values on critical fields,
# and simple business rules.

from pyspark.sql.functions import sum, when

print("Bronze count:", sales_bronze.count())
print("Silver count:", sales_silver.count())

sales_silver.select(
    sum(when(col("order_id").isNull(), 1).otherwise(0)).alias("null_order_id"),
    sum(when(col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    sum(when(col("sales").isNull(), 1).otherwise(0)).alias("null_sales")
).show()

sales_silver.filter(col("sales") < 0).show()
sales_silver.filter(col("quantity") <= 0).show()
sales_silver.filter((col("discount") < 0) | (col("discount") > 1)).show()

Bronze count: 9999
Silver count: 9986
+-------------+---------------+----------+
|null_order_id|null_product_id|null_sales|
+-------------+---------------+----------+
|            0|              0|         0|
+-------------+---------------+----------+

+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+--------------+------------+-----------+-------------+--------+
|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|ingestion_date|ingestion_ts|source_file|source_system|batch_id|
+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+--------------+------------+-----------+--------

In [14]:
# Step 5 - Reset the Silver table if needed
# This is useful during development to rebuild the Silver layer cleanly.

spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")

DataFrame[]

In [15]:
# Step 6 - Write the standardized dataset into the Silver layer as an Iceberg table

sales_silver.writeTo("nessie.silver.sales").using("iceberg").createOrReplace()

In [16]:
# Step 7 - Query the Silver table to verify that the data was written correctly

spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").show()
spark.sql("SELECT * FROM nessie.silver.sales LIMIT 10").show(truncate=False)

+--------+
|count(1)|
+--------+
|    9986|
+--------+

+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------------------------------------------------+-------+--------+--------+--------+--------------+--------------------------+--------------------+--------------+---------+
|order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name   |segment    |country      |city         |state     |postal_code|region |product_id     |category       |sub_category|product_name                                                  |sales  |quantity|discount|profit  |ingestion_date|ingestion_ts              |source_file         |source_system |batch_id |
+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+-------+---------------+-----

In [17]:
# Step 8 - Inspect the Iceberg metadata tables for the Silver sales table
# This confirms that the Silver layer is also versioned and auditable.

spark.sql("SELECT * FROM nessie.silver.sales.history").show(truncate=False)
spark.sql("SELECT * FROM nessie.silver.sales.snapshots").show(truncate=False)

+----------------------+-------------------+---------+-------------------+
|made_current_at       |snapshot_id        |parent_id|is_current_ancestor|
+----------------------+-------------------+---------+-------------------+
|2026-03-12 00:42:13.59|4621332001675844858|NULL     |true               |
+----------------------+-------------------+---------+-------------------+

+----------------------+-------------------+---------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------